## Environment Definition

## Reward Design

In [100]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

class TradingGymEnv(gym.Env):
    def __init__(
        self,
        df,
        state_features,
        transaction_cost=0.001,
        lambda_vol=0.01,
        lambda_dd=0.02,
        lambda_turnover=0.0,
        lambda_trend=0.0,
        reward_return_weight=1.0,
        reward_clip=None,
        # Optional: discourage long inactivity (mild)
        inactivity_penalty=0.0,
        inactivity_threshold=0.25,
        inactivity_window=20,
        # Optional: risk-normalized exposure
        use_risk_normalized_exposure=False,
        volatility_scale=1.0
    ):
        super().__init__()

        self.df = df.reset_index(drop=True).copy()
        self.state_features = state_features
        self.transaction_cost = transaction_cost
        self.lambda_vol = lambda_vol
        self.lambda_dd = lambda_dd
        self.lambda_turnover = lambda_turnover
        self.lambda_trend = lambda_trend
        self.reward_return_weight = reward_return_weight
        self.reward_clip = reward_clip
        self.inactivity_penalty = inactivity_penalty
        self.inactivity_threshold = inactivity_threshold
        self.inactivity_window = inactivity_window
        self.use_risk_normalized_exposure = use_risk_normalized_exposure
        self.volatility_scale = volatility_scale
        self.n_steps = len(self.df)

        required_cols = ["raw_return_1d", "regime_volatility_raw"]
        missing_cols = [c for c in required_cols if c not in self.df.columns]
        if missing_cols:
            raise ValueError(f"DataFrame is missing required columns: {missing_cols}")

        # 9 discrete target positions (short to long)
        # 0 -> -1.0, 1 -> -0.75, 2 -> -0.5, 3 -> -0.25,
        # 4 -> 0.0, 5 -> 0.25, 6 -> 0.5, 7 -> 0.75, 8 -> 1.0
        self.action_space = spaces.Discrete(9)
        self.position_levels = np.array(
            [-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0],
            dtype=np.float32
        )

        # Observation = state_features + current position
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(len(self.state_features) + 1,),
            dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.current_step = 0
        self.position = 0.0
        self.target_position = 0.0
        self.total_reward = 0.0
        self.portfolio_value = 1.0
        self.running_max = 1.0

        # Exposure tracking (diagnostics)
        self.exposure_sum = 0.0
        self.exposure_count = 0
        self.low_exposure_steps = 0

        state = self._get_state()
        info = {
            "portfolio_value": self.portfolio_value,
            "position": self.position,
            "target_position": self.target_position,
            "step": self.current_step
        }
        return state, info

    def _get_state(self):
        market_state = self.df.loc[self.current_step, self.state_features].values.astype(np.float32)
        full_state = np.append(market_state, np.float32(self.position))
        return full_state.astype(np.float32)

    def step(self, action):
        reward = 0.0

        # Stop before next-day return becomes unavailable
        if self.current_step >= self.n_steps - 2:
            terminated = True
            truncated = False
            next_state = np.zeros(len(self.state_features) + 1, dtype=np.float32)
            info = {
                "portfolio_value": self.portfolio_value,
                "position": self.position,
                "step": self.current_step
            }
            return next_state, reward, terminated, truncated, info

        action = int(action)
        target_position = float(self.position_levels[action])

        # Optional risk-normalized exposure (reduce leverage in high vol)
        current_vol_raw = float(self.df.loc[self.current_step, "regime_volatility_raw"])
        effective_position = target_position
        if self.use_risk_normalized_exposure:
            effective_position = target_position / (1 + self.volatility_scale * current_vol_raw)

        # Transaction cost depends on how much position changes
        position_change = abs(effective_position - self.position)
        trade_cost = position_change * self.transaction_cost

        # Apply new position (effective)
        self.target_position = target_position
        self.position = float(effective_position)

        # Next-day raw return drives PnL
        next_return = float(self.df.loc[self.current_step + 1, "raw_return_1d"])
        pnl = self.position * next_return

        # True net trading return (used for portfolio accounting)
        net_return = pnl - trade_cost

        # Risk penalties (use RAW volatility to avoid scaled leakage)
        current_vol = current_vol_raw
        current_drawdown = max(
            0.0,
            (self.running_max - self.portfolio_value) / (self.running_max + 1e-8)
        )

        # Trend alignment bonus (encourage participation in favorable trends)
        trend_alignment = self.position * float(np.sign(next_return))

        # Mild inactivity penalty (optional)
        if abs(self.position) < self.inactivity_threshold:
            self.low_exposure_steps += 1
        else:
            self.low_exposure_steps = 0
        inactivity_pen = self.inactivity_penalty if self.low_exposure_steps >= self.inactivity_window else 0.0

        # Training reward (shaped)
        reward = (
            self.reward_return_weight * net_return
            - self.lambda_vol * current_vol
            - self.lambda_dd * current_drawdown
            - self.lambda_turnover * position_change
            + self.lambda_trend * trend_alignment
            - inactivity_pen
        )

        if self.reward_clip is not None:
            reward = float(np.clip(reward, -self.reward_clip, self.reward_clip))
        else:
            reward = float(reward)

        # Update portfolio using TRUE net return
        self.total_reward += reward
        self.portfolio_value *= (1 + net_return)
        self.running_max = max(self.running_max, self.portfolio_value)

        # Exposure diagnostics
        self.exposure_sum += abs(self.position)
        self.exposure_count += 1
        avg_abs_exposure = self.exposure_sum / max(1, self.exposure_count)

        # Move forward
        self.current_step += 1

        terminated = self.current_step >= self.n_steps - 2
        truncated = False

        if not terminated:
            next_state = self._get_state()
        else:
            next_state = np.zeros(len(self.state_features) + 1, dtype=np.float32)

        info = {
            "portfolio_value": self.portfolio_value,
            "position": self.position,
            "target_position": self.target_position,
            "effective_position": self.position,
            "step": self.current_step,
            "position_change": position_change,
            "trade_cost": trade_cost,
            "pnl": pnl,
            "net_return": net_return,
            "reward": reward,
            "trend_alignment": trend_alignment,
            "inactivity_penalty": inactivity_pen,
            "avg_abs_exposure": avg_abs_exposure,
            "drawdown": current_drawdown
        }

        return next_state, reward, terminated, truncated, info

## Data Loading

In [101]:
import pandas as pd
import numpy as np

# ================================
# Load dataset
# ================================
df_all = pd.read_csv("global_trading_dataset.csv")

# Convert date
df_all["date"] = pd.to_datetime(df_all["date"])

# Sort by market and time
df_all = df_all.sort_values(["market", "date"]).reset_index(drop=True)

# Ensure numeric columns are float
numeric_cols = ["open", "high", "low", "close", "volume"]
df_all[numeric_cols] = df_all[numeric_cols].astype(float)

# ================================
# Create core return features
# ================================
df_all["raw_return_1d"] = df_all.groupby("market")["close"].pct_change()

df_all["return_5d"] = (
    df_all.groupby("market")["close"].pct_change(5)
)

df_all["return_10d"] = (
    df_all.groupby("market")["close"].pct_change(10)
)

# ================================
# Moving averages
# ================================
df_all["MA_5"] = (
    df_all.groupby("market")["close"]
    .transform(lambda x: x.rolling(5).mean())
)

df_all["MA_20"] = (
    df_all.groupby("market")["close"]
    .transform(lambda x: x.rolling(20).mean())
)

df_all["MA_gap"] = (df_all["MA_5"] / df_all["MA_20"]) - 1

# Trend strength feature (important for PPO)
df_all["trend_strength"] = df_all["MA_gap"]

# Bull / Bear regime indicator
df_all["bull_flag"] = (df_all["MA_5"] > df_all["MA_20"]).astype(int)

# Longer MA for bear market flag (no leakage)
df_all["MA_50"] = (
    df_all.groupby("market")["close"]
    .transform(lambda x: x.rolling(50).mean())
)

df_all["bear_market_flag"] = (df_all["close"] < df_all["MA_50"]).astype(int)

# Rolling downside return mean (negative returns only)
def rolling_downside_mean(series, window=20):
    neg = series.copy()
    neg[neg > 0] = 0.0
    return neg.rolling(window).mean()

df_all["rolling_downside_mean"] = (
    df_all.groupby("market")["raw_return_1d"]
    .transform(lambda x: rolling_downside_mean(x, 20))
)

# Trend persistence indicator (rolling sign agreement)
def trend_persistence(series, window=20):
    return np.sign(series).rolling(window).mean()

df_all["trend_persistence"] = (
    df_all.groupby("market")["raw_return_1d"]
    .transform(lambda x: trend_persistence(x, 20))
)

# ================================
# Volatility features
# ================================
# NOTE: raw volatility is used for reward (no scaling)
df_all["regime_volatility_raw"] = (
    df_all.groupby("market")["raw_return_1d"]
    .transform(lambda x: x.rolling(20).std())
)

# Scaled volatility feature for state input
# (kept separate from raw to avoid leakage into reward)
df_all["regime_volatility"] = df_all["regime_volatility_raw"]

# Short-term volatility for state input
# (uses only past data via rolling)
df_all["volatility_10"] = (
    df_all.groupby("market")["raw_return_1d"]
    .transform(lambda x: x.rolling(10).std())
)

# Downside volatility (negative returns only)
def downside_volatility(series, window=20):
    neg = series.copy()
    neg[neg > 0] = 0.0
    return neg.rolling(window).std()

df_all["downside_volatility"] = (
    df_all.groupby("market")["raw_return_1d"]
    .transform(lambda x: downside_volatility(x, 20))
)

# Rolling skewness of returns
# (uses only past data via rolling)
df_all["rolling_skewness"] = (
    df_all.groupby("market")["raw_return_1d"]
    .transform(lambda x: x.rolling(20).skew())
)

# Recent drawdown (rolling max drawdown proxy)
# (uses only past data via rolling window)
def rolling_drawdown(close_series, window=60):
    rolling_max = close_series.rolling(window).max()
    return (close_series / (rolling_max + 1e-8)) - 1.0

df_all["recent_drawdown"] = (
    df_all.groupby("market")["close"]
    .transform(lambda x: rolling_drawdown(x, 60))
)

# Moving average slope (trend acceleration proxy)
# (uses only past data via MA_20 diff)
df_all["ma_slope"] = (
    df_all.groupby("market")["MA_20"]
    .transform(lambda x: x.diff(5) / 5.0)
)

# ================================
# Volume change
# ================================
df_all["volume_change"] = (
    df_all.groupby("market")["volume"].pct_change()
)

# ================================
# RSI (14)
# ================================
def compute_rsi(series, window=14):
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()

    rs = avg_gain / (avg_loss + 1e-8)
    rsi = 100 - (100 / (1 + rs))

    return rsi

df_all["RSI_14"] = (
    df_all.groupby("market")["close"]
    .transform(lambda x: compute_rsi(x, 14))
)

# ================================
# MACD
# ================================
ema12 = df_all.groupby("market")["close"].transform(lambda x: x.ewm(span=12).mean())
ema26 = df_all.groupby("market")["close"].transform(lambda x: x.ewm(span=26).mean())

df_all["MACD"] = ema12 - ema26

# ================================
# Sentiment placeholder features
# (default to zero if not provided)
# ================================
for col in ["news_sentiment_score", "social_sentiment_score", "sentiment_change"]:
    if col not in df_all.columns:
        df_all[col] = 0.0

# ================================
# Market ID feature (for global training)
# ================================
market_id_map = {m: i for i, m in enumerate(sorted(df_all["market"].unique()))}
df_all["market_id"] = df_all["market"].map(market_id_map).astype(int)

# ================================
# Regime bucket placeholders
# (filled AFTER split using train quantiles)
# ================================
df_all["volatility_bucket"] = np.nan
df_all["trend_bucket"] = np.nan

# ================================
# Position feature placeholder
# (used by RL environment)
# ================================
df_all["position_feature"] = 0.0

# ================================
# Clean NaN / Inf values
# ================================
df_all = df_all.replace([np.inf, -np.inf], np.nan)

# Drop NaN only for continuous features (bucket placeholders are filled later)
dropna_cols = [
    "raw_return_1d",
    "return_5d",
    "return_10d",
    "MA_5",
    "MA_20",
    "MA_50",
    "MA_gap",
    "volatility_10",
    "volume_change",
    "RSI_14",
    "MACD",
    "trend_strength",
    "bull_flag",
    "bear_market_flag",
    "regime_volatility_raw",
    "regime_volatility",
    "downside_volatility",
    "rolling_downside_mean",
    "rolling_skewness",
    "trend_persistence",
    "recent_drawdown",
    "ma_slope"
]

df_all = df_all.dropna(subset=dropna_cols).reset_index(drop=True)

# ================================
# Dataset diagnostics
# ================================
print("Dataset shape:", df_all.shape)
print("Columns:", df_all.columns.tolist())

print("\nHead:")
print(df_all.head())

print("\nMarket distribution:")
print(df_all["market"].value_counts())

print("\nNaN count:")
print(df_all.isna().sum())

print("\nInf count:")
print(np.isinf(df_all.select_dtypes(include=[float, int])).sum())

Dataset shape: (7063, 37)
Columns: ['date', 'close', 'high', 'low', 'open', 'volume', 'market', 'return_1d', 'return_5d', 'return_10d', 'MA_5', 'MA_20', 'MA_gap', 'volatility_10', 'volume_change', 'RSI_14', 'MACD', 'raw_return_1d', 'trend_strength', 'bull_flag', 'MA_50', 'bear_market_flag', 'rolling_downside_mean', 'trend_persistence', 'regime_volatility_raw', 'regime_volatility', 'downside_volatility', 'rolling_skewness', 'recent_drawdown', 'ma_slope', 'news_sentiment_score', 'social_sentiment_score', 'sentiment_change', 'market_id', 'volatility_bucket', 'trend_bucket', 'position_feature']

Head:
        date        close         high          low         open    volume  \
0 2020-04-30  2860.082031  2865.590088  2832.384033  2832.384033  242500.0   
1 2020-05-06  2878.139893  2879.262939  2830.654053  2831.633057  249800.0   
2 2020-05-07  2871.522949  2882.021973  2864.581055  2876.472900  226700.0   
3 2020-05-08  2895.340088  2903.800049  2879.199951  2882.709961  226800.0   
4 202

## Feature Engineering, Split, and Scaling

In [102]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# ======================================
# State features
# ======================================
base_features = [
    "return_1d",
    "return_5d",
    "return_10d",
    "MA_gap",
    "volatility_10",
    "volume_change",
    "RSI_14",
    "MACD"
]

regime_features = [
    "trend_strength",
    "regime_volatility",
    "bull_flag"
]

extra_features = [
    "downside_volatility",
    "rolling_downside_mean",
    "rolling_skewness",
    "trend_persistence",
    "recent_drawdown",
    "ma_slope",
    "bear_market_flag"
]

bucket_features = [
    "volatility_bucket",
    "trend_bucket"
]

sentiment_features = [
    "news_sentiment_score",
    "social_sentiment_score",
    "sentiment_change"
]

id_features = [
    "market_id"
]

state_features = base_features + regime_features + extra_features + bucket_features + sentiment_features + id_features
# IMPORTANT:
# raw_return_1d is ONLY for reward / backtesting
# regime_volatility_raw is ONLY for reward penalty
# bucket/id features are discrete and must never be scaled
scale_cols = base_features + regime_features + extra_features + sentiment_features

# ======================================
# Utility: time split
# ======================================
def time_split(df, train_ratio=0.6, val_ratio=0.2):
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    df_train = df.iloc[:train_end].copy()
    df_val = df.iloc[train_end:val_end].copy()
    df_test = df.iloc[val_end:].copy()

    return df_train, df_val, df_test


def add_regime_buckets(df_train, df_val, df_test):
    # Use TRAIN quantiles only (avoid leakage)
    vol_q = df_train["regime_volatility_raw"].quantile([0.33, 0.66]).values
    trend_q = df_train["trend_strength"].quantile([0.33, 0.66]).values

    def apply_buckets(df_part):
        df_part["volatility_bucket"] = pd.cut(
            df_part["regime_volatility_raw"],
            bins=[-np.inf, vol_q[0], vol_q[1], np.inf],
            labels=[0, 1, 2]
        ).astype(int)
        df_part["trend_bucket"] = pd.cut(
            df_part["trend_strength"],
            bins=[-np.inf, trend_q[0], trend_q[1], np.inf],
            labels=[0, 1, 2]
        ).astype(int)

    apply_buckets(df_train)
    apply_buckets(df_val)
    apply_buckets(df_test)

    return df_train, df_val, df_test

# ======================================
# Prepare RL dataframe
# ======================================
df_rl = df_all.copy()

df_rl = df_rl.replace([np.inf, -np.inf], np.nan)
# Bucket features are filled AFTER split, so exclude them from dropna here
dropna_cols = [c for c in state_features if c not in bucket_features]
df_rl = df_rl.dropna(subset=dropna_cols).reset_index(drop=True)

# ======================================
# Use US market first for initial PPO testing
# ======================================
df_us = (
    df_rl[df_rl["market"] == "US"]
    .sort_values("date")
    .reset_index(drop=True)
    .copy()
)

# Split before scaling to avoid leakage
df_us_train, df_us_val, df_us_test = time_split(df_us, train_ratio=0.6, val_ratio=0.2)

# Add regime buckets using TRAIN quantiles only (avoid leakage)
df_us_train, df_us_val, df_us_test = add_regime_buckets(df_us_train, df_us_val, df_us_test)

print("US split sizes:")
print("Train:", df_us_train.shape)
print("Val  :", df_us_val.shape)
print("Test :", df_us_test.shape)

# ======================================
# Fit scaler on TRAIN only
# ======================================
scaler = StandardScaler()
df_us_train[scale_cols] = scaler.fit_transform(df_us_train[scale_cols])
df_us_val[scale_cols] = scaler.transform(df_us_val[scale_cols])
df_us_test[scale_cols] = scaler.transform(df_us_test[scale_cols])

# ======================================
# Data quality checks
# ======================================
for name, df_part in [("TRAIN", df_us_train), ("VAL", df_us_val), ("TEST", df_us_test)]:
    print(f"\n===== {name} =====")
    print("NaN count:")
    print(df_part[state_features].isna().sum())

    print("\nInf count:")
    print(np.isinf(df_part[scale_cols].select_dtypes(include=[float, int]).values).sum())

    print("\nScaled feature summary:")
    print(df_part[scale_cols].describe().T[["mean", "std", "min", "max"]])

# ======================================
# Environment sanity check (train only)
# ======================================
env = TradingGymEnv(
    df=df_us_train,
    state_features=state_features,
    transaction_cost=0.002,
    lambda_vol=0.002,
    lambda_dd=0.005,
    reward_clip=None
)

obs, info = env.reset()

print("\nInitial state:", obs)
print("Reset info:", info)
print("Observation shape:", obs.shape)
print("Any NaN in obs:", np.isnan(obs).any())
print("Any Inf in obs:", np.isinf(obs).any())

# Random rollout test using new 9-action space
done = False
while not done:
    action = np.random.choice(list(range(env.action_space.n)))
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

print("\nFinal portfolio value:", info["portfolio_value"])
print("Total reward:", env.total_reward)

US split sizes:
Train: (856, 37)
Val  : (286, 37)
Test : (286, 37)

===== TRAIN =====
NaN count:
return_1d                 0
return_5d                 0
return_10d                0
MA_gap                    0
volatility_10             0
volume_change             0
RSI_14                    0
MACD                      0
trend_strength            0
regime_volatility         0
bull_flag                 0
downside_volatility       0
rolling_downside_mean     0
rolling_skewness          0
trend_persistence         0
recent_drawdown           0
ma_slope                  0
bear_market_flag          0
volatility_bucket         0
trend_bucket              0
news_sentiment_score      0
social_sentiment_score    0
sentiment_change          0
market_id                 0
dtype: int64

Inf count:
0

Scaled feature summary:
                                mean       std       min       max
return_1d              -4.150366e-18  1.000585 -5.151581  4.744544
return_5d              -8.300733e-18  1.00058

In [103]:
import numpy as np

print("Checking RL dataset quality...")

for name, df_part in [("TRAIN", df_us_train), ("VAL", df_us_val), ("TEST", df_us_test)]:
    print(f"\n==================== {name} ====================")

    # NaN check
    print("\nNaN count per feature:")
    print(df_part[state_features].isna().sum())

    # Inf check
    print("\nInf count in scaled features:")
    print(np.isinf(df_part[scale_cols].values).sum())

    # Extreme value check
    print("\nMax absolute value per feature:")
    print(np.abs(df_part[scale_cols]).max())

    # Feature statistics
    print("\nFeature summary:")
    print(df_part[scale_cols].describe().T[["mean", "std", "min", "max"]])

    # Position feature check
    print("\nPosition feature unique values:")
    print(df_part["position_feature"].unique())

Checking RL dataset quality...

==================== TRAIN ====================

NaN count per feature:
return_1d                 0
return_5d                 0
return_10d                0
MA_gap                    0
volatility_10             0
volume_change             0
RSI_14                    0
MACD                      0
trend_strength            0
regime_volatility         0
bull_flag                 0
downside_volatility       0
rolling_downside_mean     0
rolling_skewness          0
trend_persistence         0
recent_drawdown           0
ma_slope                  0
bear_market_flag          0
volatility_bucket         0
trend_bucket              0
news_sentiment_score      0
social_sentiment_score    0
sentiment_change          0
market_id                 0
dtype: int64

Inf count in scaled features:
0

Max absolute value per feature:
return_1d                 5.151581
return_5d                 4.336059
return_10d                3.932553
MA_gap                    3.425563
volat

In [104]:
# ======================================
# Environment reset sanity checks
# ======================================

def check_env(env, name):
    print(f"\n================ {name} ENV CHECK ================")

    obs, info = env.reset()

    print("Initial obs:", obs)
    print("Obs shape:", obs.shape)
    print("Obs dtype:", obs.dtype)

    print("Action space:", env.action_space)
    print("Action space size:", env.action_space.n)

    print("Any NaN in obs:", np.isnan(obs).any())
    print("Any Inf in obs:", np.isinf(obs).any())

    print("\nReset info:", info)

    # Assertions
    assert obs.shape == (len(state_features) + 1,), f"Unexpected observation shape: {obs.shape}"
    assert env.action_space.n == 9, f"Unexpected action space: {env.action_space}"
    assert not np.isnan(obs).any(), "Observation contains NaN"
    assert not np.isinf(obs).any(), "Observation contains Inf"

    print("\nEnvironment reset check passed.")


# Train environment
env_train = TradingGymEnv(
    df=df_us_train,
    state_features=state_features,
    transaction_cost=0.002,
    lambda_vol=0.002,
    lambda_dd=0.005,
    reward_clip=None
)

# Validation environment
env_val = TradingGymEnv(
    df=df_us_val,
    state_features=state_features,
    transaction_cost=0.002,
    lambda_vol=0.002,
    lambda_dd=0.005,
    reward_clip=None
)

# Test environment
env_test = TradingGymEnv(
    df=df_us_test,
    state_features=state_features,
    transaction_cost=0.002,
    lambda_vol=0.002,
    lambda_dd=0.005,
    reward_clip=None
)

# Run checks
check_env(env_train, "TRAIN")
check_env(env_val, "VAL")
check_env(env_test, "TEST")


================ TRAIN ENV CHECK ================
Initial obs: [ 1.1525091  -0.66683525  0.3324516   1.4864349   2.2637842  -0.42739922
  1.0159045   0.44637376  1.4864349   3.9484253   0.71953493  2.3933573
 -2.4322972   0.78013027 -0.24253942 -2.6209955   1.985431   -0.63504225
  2.          2.          0.          0.          0.          4.
  0.        ]
Obs shape: (25,)
Obs dtype: float32
Action space: Discrete(9)
Action space size: 9
Any NaN in obs: False
Any Inf in obs: False

Reset info: {'portfolio_value': 1.0, 'position': 0.0, 'target_position': 0.0, 'step': 0}

Environment reset check passed.

================ VAL ENV CHECK ================
Initial obs: [-0.86453795 -0.72644764 -0.61886424 -0.34041002 -0.8804676  -0.5167997
 -1.5879464  -0.44885105 -0.34041002 -0.7971372  -1.3897866  -0.62835526
  0.5052575   0.13680762  0.22717494 -0.02038754  0.13402958  1.5746983
  0.          0.          0.          0.          0.          4.
  0.        ]
Obs shape: (25,)
Obs dtype: flo

## Training + Validation Model Selection

### Fast Hyperparameter Search (Coarse)

### Refined Training (Multi-Seed)

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
import random
import torch
import os
import hashlib

# ======================================
# Reproducibility
# ======================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# ======================================
# Temporal window (stacked observations)
# ======================================
DEV_MODE = True  # True for faster experiments
N_STACK_DEV = 10
N_STACK_FULL = 20
N_STACK = N_STACK_DEV if DEV_MODE else N_STACK_FULL
# NOTE: for full recurrent PPO, swap to sb3-contrib RecurrentPPO with LSTM policy

# ======================================
# Validation-based reward hyperparameter search
# ======================================
# Coarse grid (fast) then refine top candidates
COARSE_GRID = {
    "transaction_cost": [0.001, 0.002],
    "lambda_vol": [0.001, 0.002],
    "lambda_dd": [0.002, 0.005],
    "lambda_turnover": [0.0, 0.001],
    "lambda_trend": [0.05, 0.1],
    "reward_return_weight": [1.0, 1.5]
}

REFINE_GRID = COARSE_GRID  # keep identical by default

COARSE_TIMESTEPS = 20000 if DEV_MODE else 30000
REFINE_TIMESTEPS = 80000 if DEV_MODE else 100000

EVAL_INTERVAL = 5000
PATIENCE = 3
TOP_K = 2

# Multi-seed refinement
USE_MULTI_SEED = True
REFINE_SEEDS = [SEED, SEED + 1, SEED + 2]

# Optional penalties / exposure controls
USE_INACTIVITY_PENALTY = False
INACTIVITY_PENALTY = 0.0001
INACTIVITY_WINDOW = 20

USE_RISK_NORMALIZED_EXPOSURE = False
VOLATILITY_SCALE = 1.0


def expand_grid(grid):
    keys = list(grid.keys())
    configs = [{}]
    for k in keys:
        configs = [dict(cfg, **{k: v}) for cfg in configs for v in grid[k]]
    return configs


def build_env_kwargs(params):
    # Centralized reward/exposure configuration
    return {
        "transaction_cost": params["transaction_cost"],
        "lambda_vol": params["lambda_vol"],
        "lambda_dd": params["lambda_dd"],
        "lambda_turnover": params["lambda_turnover"],
        "lambda_trend": params["lambda_trend"],
        "reward_return_weight": params["reward_return_weight"],
        "inactivity_penalty": INACTIVITY_PENALTY if USE_INACTIVITY_PENALTY else 0.0,
        "inactivity_window": INACTIVITY_WINDOW,
        "use_risk_normalized_exposure": USE_RISK_NORMALIZED_EXPOSURE,
        "volatility_scale": VOLATILITY_SCALE
    }


def make_env(df, env_kwargs):
    env = DummyVecEnv([
        lambda: Monitor(
            TradingGymEnv(
                df=df,
                state_features=state_features,
                reward_clip=None,
                **env_kwargs
            )
        )
    ])
    return VecFrameStack(env, n_stack=N_STACK)


def run_backtest_vec(model, df_eval, env_kwargs):
    # Use the same environment + stacking as training
    eval_env = make_env(df_eval, env_kwargs)

    obs = eval_env.reset()
    done = False
    portfolio_values = [1.0]
    last_info = None
    turnover_sum = 0.0
    exposure_sum = 0.0
    steps = 0

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = eval_env.step(action)
        done = bool(done[0])
        last_info = info[0]
        portfolio_values.append(last_info["portfolio_value"])

        turnover_sum += float(last_info.get("position_change", 0.0))
        exposure_sum += abs(float(last_info.get("position", 0.0)))
        steps += 1

    avg_turnover = turnover_sum / max(1, steps)
    avg_abs_exposure = exposure_sum / max(1, steps)

    return np.array(portfolio_values), last_info, avg_turnover, avg_abs_exposure


def compute_basic_metrics(equity_curve, turnover, avg_abs_exposure):
    returns = np.diff(equity_curve) / equity_curve[:-1]
    sharpe = np.mean(returns) / (np.std(returns) + 1e-8)
    running_max = np.maximum.accumulate(equity_curve)
    drawdown = (running_max - equity_curve) / (running_max + 1e-8)
    max_dd = np.max(drawdown)
    return {
        "Sharpe": float(sharpe),
        "Max Drawdown": float(max_dd),
        "Turnover": float(turnover),
        "Avg Abs Exposure": float(avg_abs_exposure)
    }


def validation_score(metrics):
    return metrics["Sharpe"] - 0.4 * metrics["Max Drawdown"] - 0.1 * metrics["Turnover"]


def cache_key_from(params, timesteps, seed, stage):
    raw = f"{sorted(params.items())}_{timesteps}_{N_STACK}_{seed}_{stage}"
    return hashlib.md5(raw.encode("utf-8")).hexdigest()


def train_with_early_stopping(df_train, df_val, params, timesteps, seed, stage, cache):
    env_kwargs = build_env_kwargs(params)
    key = cache_key_from(params, timesteps, seed, stage)
    cache_path = os.path.join("ppo_cache_models", f"{key}.zip")

    if key in cache and os.path.exists(cache_path):
        cached = cache[key]
        model = PPO.load(cache_path, env=make_env(df_train, env_kwargs))
        return model, cached["metrics"], cached["score"], env_kwargs

    os.makedirs("ppo_cache_models", exist_ok=True)
    train_env = make_env(df_train, env_kwargs)

    model = PPO(
        policy="MlpPolicy",
        env=train_env,
        verbose=0,
        learning_rate=1e-4,
        n_steps=256,
        batch_size=64,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        seed=seed,
        tensorboard_log="./ppo_trading_tensorboard/"
    )

    best_score = -np.inf
    best_metrics = None
    best_path = cache_path
    steps = 0
    no_improve = 0
    early_stopped = False

    while steps < timesteps:
        step_size = min(EVAL_INTERVAL, timesteps - steps)
        model.learn(step_size, reset_num_timesteps=False)
        steps += step_size

        val_curve, _, val_turnover, val_avg_exp = run_backtest_vec(model, df_val, env_kwargs)
        metrics = compute_basic_metrics(val_curve, val_turnover, val_avg_exp)
        score = validation_score(metrics)

        if score > best_score + 1e-6:
            best_score = score
            best_metrics = metrics
            no_improve = 0
            model.save(best_path)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                early_stopped = True
                break

    model = PPO.load(best_path, env=train_env)
    cache[key] = {"metrics": best_metrics, "score": best_score}
    return model, best_metrics, best_score, env_kwargs, early_stopped


print("Starting PPO two-stage search on US TRAIN set...")

cache = {}
coarse_configs = expand_grid(COARSE_GRID)
coarse_results = []

print("\nStage 1 (coarse) ...")
for params in coarse_configs:
    model, metrics, score, env_kwargs, early_stopped = train_with_early_stopping(
        df_us_train,
        df_us_val,
        params,
        timesteps=COARSE_TIMESTEPS,
        seed=SEED,
        stage="coarse",
        cache=cache
    )
    coarse_results.append({
        "params": params,
        "score": score,
        "sharpe": metrics["Sharpe"],
        "mdd": metrics["Max Drawdown"],
        "turnover": metrics["Turnover"],
        "avg_abs_exposure": metrics["Avg Abs Exposure"]
    })

coarse_df = pd.DataFrame(coarse_results).sort_values("score", ascending=False)
print("\nCoarse ranking (top 5)")
print(coarse_df.head(5))

# Select top configs for refinement
refine_candidates = coarse_df.head(TOP_K)["params"].tolist()

print("\nStage 2 (refine) ...")
refine_results = []

best_score = -np.inf
best_model = None
best_params = None
best_early_stopped = None

for params in refine_candidates:
    seed_list = REFINE_SEEDS if USE_MULTI_SEED else [SEED]
    scores = []
    best_seed_score = -np.inf
    best_seed_model = None
    best_seed_metrics = None
    best_seed_early_stopped = False

    for seed in seed_list:
        model, metrics, score, env_kwargs, early_stopped = train_with_early_stopping(
            df_us_train,
            df_us_val,
            params,
            timesteps=REFINE_TIMESTEPS,
            seed=seed,
            stage="refine",
            cache=cache
        )
        scores.append(score)
        if score > best_seed_score:
            best_seed_score = score
            best_seed_model = model
            best_seed_metrics = metrics
            best_seed_early_stopped = early_stopped

    avg_score = float(np.mean(scores))
    refine_results.append({
        "params": params,
        "avg_score": avg_score,
        "scores": scores,
        "best_metrics": best_seed_metrics
    })

    if avg_score > best_score:
        best_score = avg_score
        best_model = best_seed_model
        best_params = params
        best_early_stopped = best_seed_early_stopped

refine_df = pd.DataFrame(refine_results).sort_values("avg_score", ascending=False)
print("\nRefined ranking")
print(refine_df)

print("\nBest params:", best_params)
print("Best score:", best_score)

# Use best model for downstream evaluation
model = best_model
best_env_kwargs = build_env_kwargs(best_params)

# Save model
model.save("ppo_us_train_model")

Starting PPO two-stage search on US TRAIN set...

Stage 1 (coarse) ...


## Single-Market Evaluation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

# ======================================
# Backtest function (stacked observations)
# ======================================
def run_backtest_stack(model, df_eval, env_kwargs):
    # Reuse the same env config + stacking as training
    eval_env = DummyVecEnv([
        lambda: Monitor(
            TradingGymEnv(
                df=df_eval,
                state_features=state_features,
                reward_clip=None,
                **env_kwargs
            )
        )
    ])
    eval_env = VecFrameStack(eval_env, n_stack=N_STACK)

    obs = eval_env.reset()
    done = False
    portfolio_values = [1.0]
    last_info = None
    turnover_sum = 0.0
    exposure_sum = 0.0
    steps = 0

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = eval_env.step(action)
        done = bool(done[0])
        last_info = info[0]
        portfolio_values.append(last_info["portfolio_value"])

        turnover_sum += float(last_info.get("position_change", 0.0))
        exposure_sum += abs(float(last_info.get("position", 0.0)))
        steps += 1

    avg_turnover = turnover_sum / max(1, steps)
    avg_abs_exposure = exposure_sum / max(1, steps)

    return np.array(portfolio_values), last_info, avg_turnover, avg_abs_exposure


# ======================================
# Run PPO backtest (best params)
# ======================================
val_equity, val_last_info, val_turnover, val_avg_exp = run_backtest_stack(model, df_us_val, best_env_kwargs)
test_equity, test_last_info, test_turnover, test_avg_exp = run_backtest_stack(model, df_us_test, best_env_kwargs)

print("Validation final value:", val_equity[-1])
print("Test final value:", test_equity[-1])


# ======================================
# Buy & Hold baseline
# ======================================
def buy_hold_curve(df):
    returns = df["raw_return_1d"].values
    equity = [1.0]
    for r in returns:
        equity.append(equity[-1] * (1 + r))
    return np.array(equity)


bh_val = buy_hold_curve(df_us_val)
bh_test = buy_hold_curve(df_us_test)


# ======================================
# Plot comparison
# ======================================
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(val_equity, label="PPO")
plt.plot(bh_val[:len(val_equity)], label="Buy & Hold")
plt.title("Validation Set")
plt.xlabel("Step")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(test_equity, label="PPO")
plt.plot(bh_test[:len(test_equity)], label="Buy & Hold")
plt.title("Test Set")
plt.xlabel("Step")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### JSON Signal Output (Single Market)

In [ ]:
import numpy as np
import pandas as pd

# ======================================
# Performance metrics
# ======================================

def performance_metrics(equity_curve, turnover=None, avg_abs_exposure=None):

    returns = np.diff(equity_curve) / equity_curve[:-1]

    total_return = equity_curve[-1] - 1

    volatility = np.std(returns)

    sharpe = np.mean(returns) / (volatility + 1e-8)

    running_max = np.maximum.accumulate(equity_curve)
    drawdown = (running_max - equity_curve) / running_max
    max_dd = np.max(drawdown)

    metrics = {
        "Total Return": total_return,
        "Volatility": volatility,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd
    }

    # Optional diagnostics
    if turnover is not None:
        metrics["Turnover"] = float(turnover)
    if avg_abs_exposure is not None:
        metrics["Avg Abs Exposure"] = float(avg_abs_exposure)

    return metrics


# ======================================
# Compute metrics
# ======================================

ppo_val_metrics = performance_metrics(val_equity, val_turnover, val_avg_exp)
ppo_test_metrics = performance_metrics(test_equity, test_turnover, test_avg_exp)

bh_val_metrics = performance_metrics(bh_val[:len(val_equity)])
bh_test_metrics = performance_metrics(bh_test[:len(test_equity)])


# ======================================
# Create comparison table
# ======================================

results = pd.DataFrame({
    
    "PPO Validation": ppo_val_metrics,
    "BuyHold Validation": bh_val_metrics,
    
    "PPO Test": ppo_test_metrics,
    "BuyHold Test": bh_test_metrics
    
})

print("\nPerformance Comparison")
print(results.round(4))

# JSON-style signal output (single market)
# NOTE: future integration point for Advisor Bot / MCP / Clawbot layer
trend_bucket = int(df_us_test["trend_bucket"].iloc[-1])
vol_bucket = int(df_us_test["volatility_bucket"].iloc[-1])

target_exposure = float(test_last_info["target_position"]) if test_last_info is not None else 0.0
effective_exposure = float(test_last_info.get("effective_position", test_last_info.get("position", 0.0))) if test_last_info is not None else 0.0

confidence = float(np.clip((ppo_val_metrics["Sharpe Ratio"] - ppo_val_metrics["Max Drawdown"]) / 3.0, 0.0, 1.0))

sentiment_score = 0.5 * (
    float(df_us_test["news_sentiment_score"].iloc[-1]) + float(df_us_test["social_sentiment_score"].iloc[-1])
)

signal = {
    "market": "US",
    "signal": "LONG" if target_exposure > 0.1 else ("SHORT" if target_exposure < -0.1 else "FLAT"),
    "target_exposure": target_exposure,
    "effective_exposure": effective_exposure,
    "confidence": confidence,
    "regime": f"trend_{trend_bucket}_vol_{vol_bucket}_bear_{int(df_us_test['bear_market_flag'].iloc[-1])}",
    "risk_level": "HIGH" if (vol_bucket == 2 or ppo_test_metrics["Max Drawdown"] > 0.2) else ("MEDIUM" if vol_bucket == 1 else "LOW"),
    "metrics": {
        "total_return": float(ppo_test_metrics["Total Return"]),
        "sharpe": float(ppo_test_metrics["Sharpe Ratio"]),
        "max_drawdown": float(ppo_test_metrics["Max Drawdown"]),
        "turnover": float(ppo_test_metrics.get("Turnover", 0.0)),
        "avg_abs_exposure": float(ppo_test_metrics.get("Avg Abs Exposure", 0.0))
    },
    "features": {
        "trend_strength": float(df_us_test["trend_strength"].iloc[-1]),
        "regime_volatility_raw": float(df_us_test["regime_volatility_raw"].iloc[-1]),
        "recent_drawdown": float(df_us_test["recent_drawdown"].iloc[-1]),
        "sentiment_score": sentiment_score
    }
}

print("\nJSON Signal (US)")
print(signal)


# ======================================
# Final portfolio values
# ======================================

print("\nFinal Portfolio Values")

print("PPO Validation:", val_equity[-1])
print("BuyHold Validation:", bh_val[len(val_equity)-1])

print("PPO Test:", test_equity[-1])
print("BuyHold Test:", bh_test[len(test_equity)-1])

In [ ]:
import numpy as np
import pandas as pd

# ======================================
# Performance metrics
# ======================================

def compute_metrics(equity_curve):

    returns = np.diff(equity_curve) / equity_curve[:-1]

    total_return = equity_curve[-1] - 1

    volatility = np.std(returns) * np.sqrt(252)

    sharpe = np.mean(returns) / (np.std(returns) + 1e-8) * np.sqrt(252)

    running_max = np.maximum.accumulate(equity_curve)

    drawdown = (running_max - equity_curve) / running_max

    max_drawdown = np.max(drawdown)

    return {
        "Total Return": total_return,
        "Volatility": volatility,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_drawdown
    }


# ======================================
# Compute metrics
# ======================================

ppo_val_metrics = compute_metrics(val_equity)
ppo_test_metrics = compute_metrics(test_equity)

bh_val_metrics = compute_metrics(bh_val[:len(val_equity)])
bh_test_metrics = compute_metrics(bh_test[:len(test_equity)])


# ======================================
# Results table
# ======================================

results = pd.DataFrame({

    "PPO Validation": ppo_val_metrics,
    "BuyHold Validation": bh_val_metrics,

    "PPO Test": ppo_test_metrics,
    "BuyHold Test": bh_test_metrics

})

print("\nPerformance Comparison")
print(results.round(4))


# ======================================
# Final portfolio values
# ======================================

print("\nFinal Portfolio Values")

print("PPO Validation:", val_equity[-1])
print("BuyHold Validation:", bh_val[len(val_equity)-1])

print("PPO Test:", test_equity[-1])
print("BuyHold Test:", bh_test[len(test_equity)-1])

## Five-Market Evaluation + Global Option

### Local vs Global Comparison

### JSON Signal Output (Multi-Market)

In [ ]:
# Duplicate summary removed; see Performance Comparison above.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import random
import torch

# ======================================
# Reproducibility
# ======================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# Temporal window (stacked observations)
DEV_MODE = True  # True for faster experiments
N_STACK_DEV = 10
N_STACK_FULL = 20
N_STACK = N_STACK_DEV if DEV_MODE else N_STACK_FULL

# ======================================
# Markets
# ======================================
markets = ["US", "UK", "CN", "HK", "SG"]

# Validation-based reward hyperparameter grids (coarse -> refine)
COARSE_GRID = {
    "transaction_cost": [0.001, 0.002],
    "lambda_vol": [0.001, 0.002],
    "lambda_dd": [0.002, 0.005],
    "lambda_turnover": [0.0, 0.001],
    "lambda_trend": [0.05, 0.1],
    "reward_return_weight": [1.0, 1.5]
}

REFINE_GRID = COARSE_GRID

COARSE_TIMESTEPS = 20000 if DEV_MODE else 30000
REFINE_TIMESTEPS = 80000 if DEV_MODE else 100000

EVAL_INTERVAL = 5000
PATIENCE = 3
TOP_K = 2

USE_MULTI_SEED = True
REFINE_SEEDS = [SEED, SEED + 1, SEED + 2]

USE_INACTIVITY_PENALTY = False
INACTIVITY_PENALTY = 0.0001
INACTIVITY_WINDOW = 20

USE_RISK_NORMALIZED_EXPOSURE = False
VOLATILITY_SCALE = 1.0

# Optional global training mode
USE_GLOBAL_TRAINING = False

all_results = []
all_equity_curves = {}
all_signals = []
market_splits = {}
market_splits_raw = {}
cache = {}

# Use the same feature setting as previous cells
base_features = [
    "return_1d",
    "return_5d",
    "return_10d",
    "MA_gap",
    "volatility_10",
    "volume_change",
    "RSI_14",
    "MACD"
]

regime_features = [
    "trend_strength",
    "regime_volatility",
    "bull_flag"
]

extra_features = [
    "downside_volatility",
    "rolling_downside_mean",
    "rolling_skewness",
    "trend_persistence",
    "recent_drawdown",
    "ma_slope",
    "bear_market_flag"
]

bucket_features = [
    "volatility_bucket",
    "trend_bucket"
]

sentiment_features = [
    "news_sentiment_score",
    "social_sentiment_score",
    "sentiment_change"
]

id_features = [
    "market_id"
]

state_features = base_features + regime_features + extra_features + bucket_features + sentiment_features + id_features

# IMPORTANT:
# raw_return_1d is ONLY for reward / backtesting
# regime_volatility_raw is ONLY for reward penalty
# bucket/id features are discrete and must never be scaled
scale_cols = base_features + regime_features + extra_features + sentiment_features

# ======================================
# Utility functions
# ======================================
def time_split(df, train_ratio=0.6, val_ratio=0.2):
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    df_train = df.iloc[:train_end].copy()
    df_val = df.iloc[train_end:val_end].copy()
    df_test = df.iloc[val_end:].copy()

    return df_train, df_val, df_test


def run_backtest_single_env(model, df_eval, state_features, env_kwargs):
    # Use stacked observations for temporal context (same as training)
    eval_env = DummyVecEnv([
        lambda: Monitor(
            TradingGymEnv(
                df=df_eval,
                state_features=state_features,
                reward_clip=None,
                **env_kwargs
            )
        )
    ])
    eval_env = VecFrameStack(eval_env, n_stack=N_STACK)

    obs = eval_env.reset()
    done = False

    portfolio_values = [1.0]
    last_info = None
    turnover_sum = 0.0
    exposure_sum = 0.0
    steps = 0

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = eval_env.step(action)
        done = bool(done[0])
        last_info = info[0]
        portfolio_values.append(last_info["portfolio_value"])

        turnover_sum += float(last_info.get("position_change", 0.0))
        exposure_sum += abs(float(last_info.get("position", 0.0)))
        steps += 1

    avg_turnover = turnover_sum / max(1, steps)
    avg_abs_exposure = exposure_sum / max(1, steps)

    return np.array(portfolio_values), last_info, avg_turnover, avg_abs_exposure


def buy_hold_curve(df_eval):
    returns = df_eval["raw_return_1d"].values

    equity = [1.0]
    for r in returns:
        equity.append(equity[-1] * (1 + r))

    return np.array(equity)


def compute_metrics(equity_curve, turnover=None, avg_abs_exposure=None):
    equity_curve = np.array(equity_curve)

    returns = np.diff(equity_curve) / equity_curve[:-1]

    total_return = equity_curve[-1] - 1
    volatility = np.std(returns) * np.sqrt(252)
    sharpe = np.mean(returns) / (np.std(returns) + 1e-8) * np.sqrt(252)

    running_max = np.maximum.accumulate(equity_curve)
    drawdown = (running_max - equity_curve) / running_max
    max_drawdown = np.max(drawdown)

    return total_return, volatility, sharpe, max_drawdown, turnover, avg_abs_exposure


def validation_score_local(metrics):
    return metrics["Sharpe"] - 0.4 * metrics["Max Drawdown"] - 0.1 * metrics["Turnover"]


def global_validation_score(model, global_scaled_splits, env_kwargs):
    scores = []
    for m in markets:
        df_val_g, _ = global_scaled_splits[m]
        val_curve, _, val_turnover, val_avg_exp = run_backtest_single_env(
            model, df_val_g, state_features, env_kwargs
        )
        _, _, val_sharpe, val_mdd, val_turnover, val_avg_exp = compute_metrics(
            val_curve, val_turnover, val_avg_exp
        )
        metrics = {
            "Sharpe": float(val_sharpe),
            "Max Drawdown": float(val_mdd),
            "Turnover": float(val_turnover),
            "Avg Abs Exposure": float(val_avg_exp)
        }
        scores.append(validation_score_local(metrics))

    return float(np.mean(scores))


def train_global_with_early_stopping(df_global_train, global_scaled_splits, params, timesteps, seed, stage, cache):
    env_kwargs = build_env_kwargs(params)
    key = cache_key_from(params, timesteps, seed, stage + "_global")
    cache_path = os.path.join("ppo_cache_models", f"{key}.zip")

    if key in cache and os.path.exists(cache_path):
        cached = cache[key]
        model = PPO.load(cache_path, env=make_env(df_global_train, env_kwargs))
        return model, cached["score"], env_kwargs

    os.makedirs("ppo_cache_models", exist_ok=True)
    train_env = make_env(df_global_train, env_kwargs)

    model = PPO(
        policy="MlpPolicy",
        env=train_env,
        verbose=0,
        learning_rate=1e-4,
        n_steps=256,
        batch_size=64,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        seed=seed
    )

    best_score = -np.inf
    steps = 0
    no_improve = 0

    while steps < timesteps:
        step_size = min(EVAL_INTERVAL, timesteps - steps)
        model.learn(step_size, reset_num_timesteps=False)
        steps += step_size

        avg_score = global_validation_score(model, global_scaled_splits, env_kwargs)

        if avg_score > best_score + 1e-6:
            best_score = avg_score
            no_improve = 0
            model.save(cache_path)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    model = PPO.load(cache_path, env=train_env)
    cache[key] = {"score": best_score}
    return model, best_score, env_kwargs


def build_signal(market, target_exposure, effective_exposure, val_metrics, test_metrics, trend_bucket, vol_bucket, feature_snapshot):
    # Confidence from validation performance and stability
    confidence = float(
        np.clip((val_metrics["Sharpe"] - val_metrics["Max Drawdown"]) / 3.0, 0.0, 1.0)
    )

    if target_exposure > 0.1:
        signal = "LONG"
    elif target_exposure < -0.1:
        signal = "SHORT"
    else:
        signal = "FLAT"

    regime = f"trend_{trend_bucket}_vol_{vol_bucket}_bear_{int(feature_snapshot.get('bear_market_flag', 0))}"
    risk_level = "HIGH" if (vol_bucket == 2 or test_metrics["Max Drawdown"] > 0.2) else (
        "MEDIUM" if vol_bucket == 1 else "LOW"
    )

    return {
        "market": market,
        "signal": signal,
        "target_exposure": float(target_exposure),
        "effective_exposure": float(effective_exposure),
        "confidence": confidence,
        "regime": regime,
        "risk_level": risk_level,
        "metrics": {
            "total_return": float(test_metrics["Total Return"]),
            "sharpe": float(test_metrics["Sharpe"]),
            "max_drawdown": float(test_metrics["Max Drawdown"]),
            "turnover": float(test_metrics["Turnover"]),
            "avg_abs_exposure": float(test_metrics["Avg Abs Exposure"])
        },
        "features": {
            "trend_strength": float(feature_snapshot.get("trend_strength", 0.0)),
            "regime_volatility_raw": float(feature_snapshot.get("regime_volatility_raw", 0.0)),
            "recent_drawdown": float(feature_snapshot.get("recent_drawdown", 0.0)),
            "sentiment_score": float(feature_snapshot.get("sentiment_score", 0.0))
        }
    }


# ======================================
# Market-by-market training and testing
# ======================================
for market_name in markets:
    print(f"\n========== Training PPO for {market_name} ==========")

    # Prepare market-specific data
    df_market = (
        df_all[df_all["market"] == market_name]
        .sort_values("date")
        .reset_index(drop=True)
        .copy()
    )

    # Clean data
    df_market = df_market.replace([np.inf, -np.inf], np.nan)
    # Bucket features are filled AFTER split, so exclude them from dropna here
    dropna_cols = [c for c in state_features if c not in bucket_features]
    df_market = df_market.dropna(subset=dropna_cols).reset_index(drop=True)

    # Train / Val / Test split BEFORE scaling
    df_train, df_val, df_test = time_split(df_market, train_ratio=0.6, val_ratio=0.2)

    # Add regime buckets using TRAIN quantiles only (avoid leakage)
    df_train, df_val, df_test = add_regime_buckets(df_train, df_val, df_test)

    # Store raw splits for optional global training
    market_splits_raw[market_name] = (df_train.copy(), df_val.copy(), df_test.copy())

    print(f"{market_name} split sizes -> Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

    # Scale train only, then transform val / test
    scaler = StandardScaler()
    df_train[scale_cols] = scaler.fit_transform(df_train[scale_cols])
    df_val[scale_cols] = scaler.transform(df_val[scale_cols])
    df_test[scale_cols] = scaler.transform(df_test[scale_cols])

    # Store split for optional global training
    market_splits[market_name] = (df_train.copy(), df_val.copy(), df_test.copy())

    # Hyperparameter search (coarse -> refine)
    coarse_configs = expand_grid(COARSE_GRID)
    coarse_results = []

    for params in coarse_configs:
        model, metrics, score, env_kwargs, early_stopped = train_with_early_stopping(
            df_train,
            df_val,
            params,
            timesteps=COARSE_TIMESTEPS,
            seed=SEED,
            stage=f"{market_name}_coarse",
            cache=cache
        )
        coarse_results.append({
            "params": params,
            "score": score,
            "sharpe": metrics["Sharpe"],
            "mdd": metrics["Max Drawdown"],
            "turnover": metrics["Turnover"],
            "avg_abs_exposure": metrics["Avg Abs Exposure"]
        })

    coarse_df = pd.DataFrame(coarse_results).sort_values("score", ascending=False)
    refine_candidates = coarse_df.head(TOP_K)["params"].tolist()

    best_score = -np.inf
    best_model = None
    best_params = None

    for params in refine_candidates:
        seed_list = REFINE_SEEDS if USE_MULTI_SEED else [SEED]
        scores = []
        best_seed_score = -np.inf
        best_seed_model = None

        for seed in seed_list:
            model, metrics, score, env_kwargs, early_stopped = train_with_early_stopping(
                df_train,
                df_val,
                params,
                timesteps=REFINE_TIMESTEPS,
                seed=seed,
                stage=f"{market_name}_refine",
                cache=cache
            )
            scores.append(score)
            if score > best_seed_score:
                best_seed_score = score
                best_seed_model = model

        avg_score = float(np.mean(scores))
        if avg_score > best_score:
            best_score = avg_score
            best_model = best_seed_model
            best_params = params

    print("Best params:", best_params)
    print("Best score:", best_score)

    # Use best model for downstream evaluation
    model = best_model
    best_env_kwargs = build_env_kwargs(best_params)

    # Save model
    model.save(f"ppo_trading_model_{market_name}")

    # Validation backtest
    val_ppo_curve, val_last_info, val_turnover, val_avg_exp = run_backtest_single_env(
        model, df_val, state_features, best_env_kwargs
    )
    val_bh_curve = buy_hold_curve(df_val)

    # Test backtest
    test_ppo_curve, test_last_info, test_turnover, test_avg_exp = run_backtest_single_env(
        model, df_test, state_features, best_env_kwargs
    )
    test_bh_curve = buy_hold_curve(df_test)

    # Store TEST curves for final plotting
    all_equity_curves[market_name] = {
        "ppo": test_ppo_curve,
        "buy_hold": test_bh_curve[:len(test_ppo_curve)]
    }

    # Compute metrics
    val_ppo_total, val_ppo_vol, val_ppo_sharpe, val_ppo_mdd, val_ppo_turnover, val_ppo_avg_exp = compute_metrics(
        val_ppo_curve, val_turnover, val_avg_exp
    )
    val_bh_total, val_bh_vol, val_bh_sharpe, val_bh_mdd, _, _ = compute_metrics(
        val_bh_curve[:len(val_ppo_curve)]
    )

    test_ppo_total, test_ppo_vol, test_ppo_sharpe, test_ppo_mdd, test_ppo_turnover, test_ppo_avg_exp = compute_metrics(
        test_ppo_curve, test_turnover, test_avg_exp
    )
    test_bh_total, test_bh_vol, test_bh_sharpe, test_bh_mdd, _, _ = compute_metrics(
        test_bh_curve[:len(test_ppo_curve)]
    )

    # Build JSON-style signal output (confidence from validation)
    trend_bucket = int(df_test["trend_bucket"].iloc[-1])
    vol_bucket = int(df_test["volatility_bucket"].iloc[-1])

    target_exposure = float(test_last_info["target_position"]) if test_last_info is not None else 0.0
    effective_exposure = float(test_last_info.get("effective_position", test_last_info.get("position", 0.0))) if test_last_info is not None else 0.0

    val_metrics = {
        "Sharpe": float(val_ppo_sharpe),
        "Max Drawdown": float(val_ppo_mdd),
        "Turnover": float(val_ppo_turnover),
        "Avg Abs Exposure": float(val_ppo_avg_exp)
    }

    test_metrics = {
        "Total Return": float(test_ppo_total),
        "Sharpe": float(test_ppo_sharpe),
        "Max Drawdown": float(test_ppo_mdd),
        "Turnover": float(test_ppo_turnover),
        "Avg Abs Exposure": float(test_ppo_avg_exp)
    }

    sentiment_score = 0.5 * (
        float(df_test["news_sentiment_score"].iloc[-1]) + float(df_test["social_sentiment_score"].iloc[-1])
    )

    feature_snapshot = {
        "trend_strength": float(df_test["trend_strength"].iloc[-1]),
        "regime_volatility_raw": float(df_test["regime_volatility_raw"].iloc[-1]),
        "recent_drawdown": float(df_test["recent_drawdown"].iloc[-1]),
        "sentiment_score": sentiment_score,
        "bear_market_flag": int(df_test["bear_market_flag"].iloc[-1])
    }

    signal = build_signal(
        market_name,
        target_exposure,
        effective_exposure,
        val_metrics,
        test_metrics,
        trend_bucket,
        vol_bucket,
        feature_snapshot
    )
    all_signals.append(signal)

    all_results.append({
        "Market": market_name,

        "PPO Val Total Return": val_ppo_total,
        "BH Val Total Return": val_bh_total,
        "PPO Val Volatility": val_ppo_vol,
        "BH Val Volatility": val_bh_vol,
        "PPO Val Sharpe": val_ppo_sharpe,
        "BH Val Sharpe": val_bh_sharpe,
        "PPO Val Max Drawdown": val_ppo_mdd,
        "BH Val Max Drawdown": val_bh_mdd,
        "PPO Val Turnover": val_ppo_turnover,
        "PPO Val Avg Abs Exposure": val_ppo_avg_exp,

        "PPO Test Total Return": test_ppo_total,
        "BH Test Total Return": test_bh_total,
        "PPO Test Volatility": test_ppo_vol,
        "BH Test Volatility": test_bh_vol,
        "PPO Test Sharpe": test_ppo_sharpe,
        "BH Test Sharpe": test_bh_sharpe,
        "PPO Test Max Drawdown": test_ppo_mdd,
        "BH Test Max Drawdown": test_bh_mdd,
        "PPO Test Turnover": test_ppo_turnover,
        "PPO Test Avg Abs Exposure": test_ppo_avg_exp
    })

print("\nAll markets training completed.")

# ======================================
# Optional global training mode
# ======================================
all_results_global = []
all_equity_curves_global = {}
all_signals_global = []

if USE_GLOBAL_TRAINING:
    print("\n========== Training GLOBAL PPO model ==========")

    # Concatenate raw TRAIN splits (avoid leakage)
    df_global_train = pd.concat([market_splits_raw[m][0] for m in markets], axis=0)
    df_global_train = df_global_train.sort_values(["date", "market"]).reset_index(drop=True)

    # Fit scaler on GLOBAL TRAIN only
    scaler_global = StandardScaler()
    df_global_train[scale_cols] = scaler_global.fit_transform(df_global_train[scale_cols])

    # Prepare scaled per-market VAL/TEST using global scaler
    global_scaled_splits = {}
    for m in markets:
        df_val_g = market_splits_raw[m][1].copy()
        df_test_g = market_splits_raw[m][2].copy()
        df_val_g[scale_cols] = scaler_global.transform(df_val_g[scale_cols])
        df_test_g[scale_cols] = scaler_global.transform(df_test_g[scale_cols])
        global_scaled_splits[m] = (df_val_g, df_test_g)

    coarse_configs = expand_grid(COARSE_GRID)
    coarse_results = []

    for params in coarse_configs:
        model, score, env_kwargs = train_global_with_early_stopping(
            df_global_train,
            global_scaled_splits,
            params,
            timesteps=COARSE_TIMESTEPS,
            seed=SEED,
            stage="global_coarse",
            cache=cache
        )
        coarse_results.append({"params": params, "score": score})

    coarse_df = pd.DataFrame(coarse_results).sort_values("score", ascending=False)
    refine_candidates = coarse_df.head(TOP_K)["params"].tolist()

    best_score = -np.inf
    best_model = None
    best_params = None
    best_env_kwargs = None

    for params in refine_candidates:
        seed_list = REFINE_SEEDS if USE_MULTI_SEED else [SEED]
        scores = []
        best_seed_score = -np.inf
        best_seed_model = None
        best_seed_env_kwargs = None

        for seed in seed_list:
            model, score, env_kwargs = train_global_with_early_stopping(
                df_global_train,
                global_scaled_splits,
                params,
                timesteps=REFINE_TIMESTEPS,
                seed=seed,
                stage="global_refine",
                cache=cache
            )
            scores.append(score)
            if score > best_seed_score:
                best_seed_score = score
                best_seed_model = model
                best_seed_env_kwargs = env_kwargs

        avg_score = float(np.mean(scores))
        if avg_score > best_score:
            best_score = avg_score
            best_model = best_seed_model
            best_params = params
            best_env_kwargs = best_seed_env_kwargs

    print("Global best params:", best_params)
    print("Global best score:", best_score)

    # Evaluate global model per market (VAL/TEST)
    for m in markets:
        df_val_g, df_test_g = global_scaled_splits[m]

        val_curve, val_last_info, val_turnover, val_avg_exp = run_backtest_single_env(
            best_model, df_val_g, state_features, best_env_kwargs
        )
        test_curve, test_last_info, test_turnover, test_avg_exp = run_backtest_single_env(
            best_model, df_test_g, state_features, best_env_kwargs
        )

        val_total, val_vol, val_sharpe, val_mdd, val_turnover, val_avg_exp = compute_metrics(
            val_curve, val_turnover, val_avg_exp
        )
        test_total, test_vol, test_sharpe, test_mdd, test_turnover, test_avg_exp = compute_metrics(
            test_curve, test_turnover, test_avg_exp
        )

        all_equity_curves_global[m] = {
            "ppo": test_curve
        }

        all_results_global.append({
            "Market": m,
            "PPO Val Total Return": val_total,
            "PPO Val Volatility": val_vol,
            "PPO Val Sharpe": val_sharpe,
            "PPO Val Max Drawdown": val_mdd,
            "PPO Val Turnover": val_turnover,
            "PPO Val Avg Abs Exposure": val_avg_exp,
            "PPO Test Total Return": test_total,
            "PPO Test Volatility": test_vol,
            "PPO Test Sharpe": test_sharpe,
            "PPO Test Max Drawdown": test_mdd,
            "PPO Test Turnover": test_turnover,
            "PPO Test Avg Abs Exposure": test_avg_exp
        })

        # Build JSON-style signal (confidence from validation)
        trend_bucket = int(df_test_g["trend_bucket"].iloc[-1])
        vol_bucket = int(df_test_g["volatility_bucket"].iloc[-1])

        target_exposure = float(test_last_info["target_position"]) if test_last_info is not None else 0.0
        effective_exposure = float(test_last_info.get("effective_position", test_last_info.get("position", 0.0))) if test_last_info is not None else 0.0

        val_metrics = {
            "Sharpe": float(val_sharpe),
            "Max Drawdown": float(val_mdd),
            "Turnover": float(val_turnover),
            "Avg Abs Exposure": float(val_avg_exp)
        }

        test_metrics = {
            "Total Return": float(test_total),
            "Sharpe": float(test_sharpe),
            "Max Drawdown": float(test_mdd),
            "Turnover": float(test_turnover),
            "Avg Abs Exposure": float(test_avg_exp)
        }

        sentiment_score = 0.5 * (
            float(df_test_g["news_sentiment_score"].iloc[-1]) + float(df_test_g["social_sentiment_score"].iloc[-1])
        )

        feature_snapshot = {
            "trend_strength": float(df_test_g["trend_strength"].iloc[-1]),
            "regime_volatility_raw": float(df_test_g["regime_volatility_raw"].iloc[-1]),
            "recent_drawdown": float(df_test_g["recent_drawdown"].iloc[-1]),
            "sentiment_score": sentiment_score,
            "bear_market_flag": int(df_test_g["bear_market_flag"].iloc[-1])
        }

        signal = build_signal(
            m,
            target_exposure,
            effective_exposure,
            val_metrics,
            test_metrics,
            trend_bucket,
            vol_bucket,
            feature_snapshot
        )
        all_signals_global.append(signal)

    # Comparison table (local vs global)
    local_df = pd.DataFrame(all_results)
    global_df = pd.DataFrame(all_results_global)

    comparison = local_df[[
        "Market",
        "PPO Test Total Return",
        "PPO Test Sharpe",
        "PPO Test Max Drawdown",
        "PPO Test Turnover",
        "PPO Test Avg Abs Exposure"
    ]].merge(
        global_df[[
            "Market",
            "PPO Test Total Return",
            "PPO Test Sharpe",
            "PPO Test Max Drawdown",
            "PPO Test Turnover",
            "PPO Test Avg Abs Exposure"
        ]],
        on="Market",
        suffixes=(" Local", " Global")
    )

    print("\nLocal vs Global (Test) Comparison")
    print(comparison.round(4))

# ======================================
# JSON signal output (local)
# NOTE: future integration point for Advisor Bot / MCP / Clawbot layer
# ======================================
print("\nJSON Signals (Local Models)")
for s in all_signals:
    print(s)

if USE_GLOBAL_TRAINING:
    print("\nJSON Signals (Global Model)")
    for s in all_signals_global:
        print(s)


In [ ]:
# ======================================
# Convert results to DataFrame
# ======================================

results_df = pd.DataFrame(all_results)

print("\nFull Results Table")
print(results_df.round(4))


# ======================================
# Test-set summary (main report table)
# ======================================

test_results = results_df[[
    "Market",
    "PPO Test Total Return",
    "BH Test Total Return",
    "PPO Test Volatility",
    "BH Test Volatility",
    "PPO Test Sharpe",
    "BH Test Sharpe",
    "PPO Test Max Drawdown",
    "BH Test Max Drawdown"
]].copy()

print("\n===== TEST SET PERFORMANCE =====")
print(test_results.round(4))


# ======================================
# Sharpe ranking (useful diagnostic)
# ======================================

ranking = test_results.sort_values("PPO Test Sharpe", ascending=False)

print("\n===== PPO Sharpe Ranking =====")
print(ranking[["Market", "PPO Test Sharpe"]].round(4))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ======================================
# Helper: compute drawdown
# ======================================
def compute_drawdown(equity):
    equity = np.array(equity)
    running_max = np.maximum.accumulate(equity)
    drawdown = (running_max - equity) / running_max
    return drawdown


# ======================================
# Equity curves (all markets)
# ======================================
fig, axes = plt.subplots(len(markets), 1, figsize=(10, 4 * len(markets)))

for i, market_name in enumerate(markets):

    ax = axes[i]

    ppo_curve = all_equity_curves[market_name]["ppo"]
    bh_curve = all_equity_curves[market_name]["buy_hold"]

    ax.plot(ppo_curve, label="PPO")
    ax.plot(bh_curve, label="Buy & Hold")

    ax.set_title(f"{market_name} Market: PPO vs Buy & Hold")
    ax.set_xlabel("Step")
    ax.set_ylabel("Portfolio Value")
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()


# ======================================
# Drawdown curves
# ======================================
fig, axes = plt.subplots(len(markets), 1, figsize=(10, 4 * len(markets)))

for i, market_name in enumerate(markets):

    ax = axes[i]

    ppo_curve = all_equity_curves[market_name]["ppo"]
    bh_curve = all_equity_curves[market_name]["buy_hold"]

    ppo_dd = compute_drawdown(ppo_curve)
    bh_dd = compute_drawdown(bh_curve)

    ax.plot(ppo_dd, label="PPO Drawdown")
    ax.plot(bh_dd, label="Buy & Hold Drawdown")

    ax.set_title(f"{market_name} Market Drawdown")
    ax.set_xlabel("Step")
    ax.set_ylabel("Drawdown")
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

## End Summary

In [ ]:
# ======================================
# End Summary (single-market run)
# ======================================
try:
    print("\nBest params:", best_params)
    print("Best validation score:", best_score)
    print("Early stopping triggered:", best_early_stopped)

    print("\nValidation metrics:")
    print(ppo_val_metrics)

    print("\nTest metrics:")
    print(ppo_test_metrics)

    avg_exp = ppo_test_metrics.get("Avg Abs Exposure", None)
    if avg_exp is not None:
        print("\nAvg abs exposure:", avg_exp)
        if avg_exp < 0.25:
            print("Note: potential underexposure (avg abs exposure < 0.25).")
except NameError:
    print("Summary skipped: run training/evaluation cells first.")
